# Document-level RE & KG System — Colab Pipeline

DocRED 기반 3단계 Incremental Stacking 파이프라인 실행 노트북.

**구조**: Stage 1 (Baseline) → Stage 2 (ATLOP+DREEAM) → Stage 3 (GAIN-lite GNN)

## 0. 환경 설정

In [ ]:
# Colab에서 실행 시 프로젝트 업로드 후:
# !unzip docre-kg.zip
# %cd docre-kg

!pip install -r requirements.txt -q

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. 데이터 다운로드 & 확인

In [ ]:
# DocRED 데이터 다운로드 (HuggingFace)
# TODO: 데이터 다운로드 스크립트 완성

# !mkdir -p data/docred data/meta
# !wget -O data/docred/train_annotated.json <URL>
# !wget -O data/docred/dev.json <URL>
# !wget -O data/meta/rel2id.json <URL>

print('데이터 파일 확인:')
!ls -la data/docred/ 2>/dev/null || echo 'data/docred/ 폴더 없음 — 데이터를 먼저 다운로드하세요'

## 2. Stage 1 — Baseline RE 학습

In [ ]:
# ── Config 로드 ──
from src.utils import load_config, set_seed

config = load_config('configs/stage1.yaml')
set_seed(config['experiment']['seed'])
print(f"Stage: {config['experiment']['stage']}")
print(f"Pooling: {config['entity_repr']['pooling']}")
print(f"Classifier: {config['relation_head']['classifier_type']}")

In [ ]:
# ── 학습 실행 ──
!python scripts/train.py --config configs/stage1.yaml

## 3. Stage 2 — ATLOP + DREEAM 학습

In [ ]:
# ── Stage 2 Config 확인 ──
config2 = load_config('configs/stage2.yaml')
print(f"Stage: {config2['experiment']['stage']}")
print(f"Pooling: {config2['entity_repr']['pooling']}")
print(f"Classifier: {config2['relation_head']['classifier_type']}")
print(f"Evidence: {config2['relation_head']['use_evidence_head']}")
print(f"Lambda: {config2.get('evidence', {}).get('lambda_evidence', 'N/A')}")

In [ ]:
!python scripts/train.py --config configs/stage2.yaml

## 4. Stage 3 — GAIN-lite GNN 학습

In [ ]:
# ── Stage 3 Config 확인 ──
config3 = load_config('configs/stage3.yaml')
print(f"Stage: {config3['experiment']['stage']}")
print(f"GNN: {config3['graph_encoder']['enabled']}")
print(f"GNN type: {config3['graph_encoder']['gnn_type']}")
print(f"GNN layers: {config3['graph_encoder']['num_layers']}")
print(f"Load from: {config3['training'].get('load_checkpoint', 'None')}")

In [ ]:
!python scripts/train.py --config configs/stage3.yaml

## 5. 평가 & 비교

In [ ]:
# ── 각 Stage 평가 ──
# !python scripts/evaluate.py --config configs/stage1.yaml --checkpoint checkpoints/stage1/best_model.pt
# !python scripts/evaluate.py --config configs/stage2.yaml --checkpoint checkpoints/stage2/best_model.pt
# !python scripts/evaluate.py --config configs/stage3.yaml --checkpoint checkpoints/stage3/best_model.pt

print('TODO: 각 Stage 체크포인트로 evaluate.py 실행')

## 6. 모델 구조 확인 (디버깅용)

In [ ]:
from src.model import DocREModel
from src.utils import count_parameters

for stage_name in ['stage1', 'stage2', 'stage3']:
    cfg = load_config(f'configs/{stage_name}.yaml')
    model = DocREModel(cfg)
    n_params = count_parameters(model)
    has_gnn = model.graph_encoder is not None
    print(f'{stage_name}: {n_params:>12,} params | GNN: {has_gnn} | Pooling: {cfg["entity_repr"]["pooling"]}')